# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a dataset defined in the Croissant schema format using the `mlcroissant` library. It follows a template to ensure consistency and reproducibility for similar datasets.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get record sets (each as an mlcroissant.RecordSet object)
record_sets = metadata.record_set
if not record_sets:
    print("No record sets found. Please ensure the schema provides accessible record sets.")
else:
    for rs in record_sets:
        print(f"RecordSet name: {rs.name} | @id: {rs['@id']}")
        print("Fields:")
        for field in rs.field:
            print(f"  - {field['@id']} ({field.name})")
        print("Data sample:")
        for record in dataset.records(record_set=rs['@id']):
            print(record)
            break # Show only first record
        print("-----")

## 3. Data Extraction
Load data from specific record sets into DataFrames for further analysis. Reference record set and field `@id`s from the overview above.

In [ ]:
# Collect available record set IDs for extraction
record_set_ids = [rs['@id'] for rs in metadata.record_set]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display columns and first five records for each record set
for rs_id in record_set_ids:
    print(f"\nRecordSet '@id': {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, and group by key demographic or assessment variables.

In [ ]:
# Example: Use the main survey record set for EDA
main_rs_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[main_rs_id]

# Select a numeric symptom score field by @id (e.g., GAD-7 score)
# Find field name for demonstration (replace with actual field @id from previous overview)
numeric_field_id = None
for field in metadata.record_set[0].field:
    if 'gad7_score' in field.name.lower() or 'phq9_score' in field.name.lower() or 'psq_score' in field.name.lower():
        numeric_field_id = field['@id']
        break
# Fallback demo if not found:
if numeric_field_id is None:
    numeric_field_id = main_df.select_dtypes(include='number').columns[0] if len(main_df.select_dtypes(include='number').columns) > 0 else main_df.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter records where the score exceeds a threshold
threshold = 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field in the filtered data
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group the filtered data by a demographic variable, e.g., village or gender by @id
group_field_id = None
for field in metadata.record_set[0].field:
    if 'village' in field.name.lower() or 'gender' in field.name.lower():
        group_field_id = field['@id']
        break
# Fallback demo if not found:
if group_field_id is None:
    demo_cols = [col for col in main_df.columns if col != numeric_field_id]
    group_field_id = demo_cols[0] if demo_cols else main_df.columns[0]
print(f"Grouping by: {group_field_id}")

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships in the dataset. Here we plot the normalized GAD-7 (or other chosen score) grouped by village or gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of normalized score
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
plt.title(f"Distribution of Normalized {numeric_field_id}")
plt.xlabel(f"Normalized {numeric_field_id}")
plt.ylabel("Count")
plt.show()

# Plot group means if grouping field is available
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,4))
    sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(f"{group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Using `mlcroissant`, we loaded a FAIR²-structured mental health survey dataset, inspected record sets and fields by their `@id`, performed filtering and normalization on symptom scores, and grouped data by key demographic variables. This notebook offers a reproducible template for AI-ready analysis of survey datasets defined by Croissant schema.